## Setup

In [1]:
from agent_base.providers.anthropic.formatters import AnthropicMessageFormatter
from agent_base.core.messages import *
from agent_base.providers.anthropic.anthropic_agent import AnthropicLLMConfig
from agent_base.providers.anthropic.provider import AnthropicProvider
import anthropic
from dotenv import load_dotenv
from pprint import pprint

load_dotenv("../../../.env")

formatter = AnthropicMessageFormatter()
client = anthropic.AsyncAnthropic()
anthropic_provider = AnthropicProvider(
    client=client,
    formatter=formatter
)

## Provider Round Trip Tests

Cannonical Message -> format -> anthropic client -> parse -> Cannonical Response

### Basic - No Tools

In [4]:
canonical_message = Message(
    role=Role.USER,
    content=[
        TextContent(
            text="Hello, world!"
        )
    ]
)
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[canonical_message],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
    )
)
pprint(res.to_dict())

2026-03-01T15:14:10.529564Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'Eq8CCkYICxgCKkDn5mYEXq3+jAq1R02/oqFfUlcUyM65MwB2s0z/wtf1s3VhRtnmYruFI35EOIwZINW+wxXh3cZAKzIuvjYVb+hMEgwkz6JefuBY5V4ajzQaDI50uSdrBPcgo/C8ECIwvPr8qtZvOnpsef6DuVhnuNYwSIkwAHq87VeM3oaGI9S2Vm789AFSGN6hqu0BJzB9KpYBezPs/HLSQ7FpYLf/E2h2aEjy3mm5m7Auuq8UKzhC8EjFyIJutw1OvxwHoUUKVjlJu4RK24YGmAvZCVrH5fav4ZzgoXP2HXQAcp9snr2d4QTQZvpzOGTreRSrZGWsJc4rI36+YzCpeUnU6hLsGBSe+awMLVycZju93+57DFZLCqGFccGHIouvqYcMxAZgjnKJezhUo9t+GAE=',
              'thinking': 'The user has greeted me with a classic "Hello, '
                          'world!" message. This is a friendly greeting and I '
                          'should respond warmly and helpfully.'},
             {'content_block_type': 'text',
              'kwargs': {},
              'text': "Hello! 👋 It's nice to meet you. How can I help you "
                      'today?'}],
 'id': '9dffa916-fea4-47cc-bc28-508ed497c79c',
 'model': 'claude-haiku-4-5-2

### Server Tools, Betas, Context Management

In [5]:
res = await anthropic_provider.generate(
    system_prompt="Be brief.",
    messages=[Message.user("What is Anthropic's latest model?")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[
            {"type": "web_search_20250305", "name": "web_search", "max_uses": 3},
        ],
        beta_headers=["files-api-2025-04-14"],
    )
)
pprint(res.to_dict())

2026-03-02T02:55:26.030657Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EtsDCkYICxgCKkBV7nChyZpYsJTNt7OU7ucXLdYCVO/qGhyLRxJyElD7PKcYwfOv/5l0hTFMrlypjk35JvQobxwzMglJNjgaApG5Egy/32ySTtQ1jkK23yEaDIskFAYC0HCn+YjZAiIw3hBzJjx5g06kZ7tP6GKQuuDuyxsUjs+XvthLBVCRG3R9UWO2UrDtGKEN+xgZjUHPKsICxjyqYuWKyBbx0Pl2uqQ+MAisY1lHzM/R3DsfqSJ7FmGCtZspAIX1oMW2RPFV8kzN37TSsI3pH9hw0NjVA/uAHg65/IvRX3UWaqRwJlcWimcUtoJHBhBI8Sl68wRStiv7X+A8+ZBKT9DCXIO8i+kwj1wqhdWHnWf+LIvLvXAaIw+jSGorsf3bFP+/amsUKvora6WQ2tC3A/Pih2EFvFtAAzVL5iTlC7lGlfBuzjb1HVmPrN9lXkGjhmiKEx2AJH87NSEfI4V19YKC+gcsJT+HbhMi8mxf8L2jlSz5yPwrDx5uj0SkJj6E4BIfSU8eq6Et8XybG+i6qFuiWXGKW9mgAx5CYSssW1dKS6YMI/5WJ+gqn6pHlU+rfhm2Hd5KN9WkFetEwgaHuDiDH8X4dUqGLEIKlDXfe3B6VA0CK4XnZELfkhgB',
              'thinking': "The user is asking about Anthropic's latest model. "
                          "My knowledge cutoff is April 2024, and today's date "
                          'is March 2, 2026. There could have been new models '
            

### Client Tools

In [6]:
from agent_base.tools.tool_types import ToolSchema
from agent_base.core.types import ToolResultContent, ContentBlockType

schema = ToolSchema(
    name="get_weather",
    description="Get current weather for a city.",
    input_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
)

msg = Message.user("What's the weather in Paris?")
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[msg],
    tool_schemas=[schema],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.to_dict())

tool_call = next(b for b in res.content if b.content_block_type == ContentBlockType.TOOL_USE)
tool_result_msg = Message(role=Role.USER, content=[
    ToolResultContent(tool_name=tool_call.tool_name, tool_id=tool_call.tool_id, tool_result="Sunny, 22°C")
])
res2 = await anthropic_provider.generate(
    system_prompt=None,
    messages=[msg, res, tool_result_msg],
    tool_schemas=[schema],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res2.to_dict())

2026-03-02T02:56:29.577703Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'EoADCkYICxgCKkCjVsXDQ9X5b5eLjVDjiIyvdJFIhVVcnpp3UijRlM6IochGSzrwAyaAB6xCeU58NShVR6d2uiGpyqHvPc1neXXvEgx2xrWqZTpNlVhJeIAaDFtlcAwmObSC75rXtiIwsVv3pwB5QWuOonu2t8Xkp5IblMUxjN5R/eWc4uGH4q9mNL66PazrRyVvR+ef9p7eKucBg0pcitl/V7IHNmuErSdsf7SgTgeLcRvdugA0o/v9E0UprqrboHR/lKXDxoM6I4FCWvtm8vz9V3qExMm+weV4ickedHSBFZlRNK6RzowFUekHC0YN2IGSTmTNvL1D9Fy9m1KAwciCAIEpQgqyb9xUy/MGz07QquJ99WYmbmq2xpo18/ilwvPB0e1ZkwmxHkWqIrZHQdVnQBV7BmryNm0HfYk5nA/zJkMSRBYpgLFvmt2xAtYtBbuk4U3SHX0sbFIS7H/zgDn2Qm5ctgMrIxzZcrm6zqbqYtBQnj+iP2/i+B3/ujnwtGIVGAE=',
              'thinking': 'The user is asking for the weather in Paris. I have '
                          'access to a `get_weather` function that takes a '
                          'city name as a parameter. This is straightforward - '
                          'I need to call the function with "Paris" as the '
                          'city parameter.'},
           

2026-03-02T02:56:30.351273Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'text',
              'kwargs': {},
              'text': 'The weather in Paris is currently **sunny** with a '
                      "temperature of **22°C** (about 72°F). It's a beautiful "
                      'day!'}],
 'id': '653fa614-17cc-4cc8-8069-404cb9bb3752',
 'model': 'claude-haiku-4-5-20251001',
 'provider': 'anthropic',
 'role': 'assistant',
 'stop_reason': 'end_turn',
 'usage': {'cache_read_tokens': 0,
           'cache_write_tokens': 0,
           'input_tokens': 724,
           'output_tokens': 34,
           'raw_usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0,
                                            'ephemeral_5m_input_tokens': 0},
                         'cache_creation_input_tokens': 0,
                         'cache_read_input_tokens': 0,
                         'inference_geo': 'not_available',
                         'input_tokens': 724,
                         'output_tokens': 34,
                         

### Server Side File Generation

In [7]:
res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message.user("Write and run a Python script that prints 'hello world'")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        beta_headers=["code-execution-2025-08-25", "files-api-2025-04-14"],
    )
)
pprint(res.to_dict())

2026-03-02T02:57:02.856847Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'ErkDCkYICxgCKkBdjjH6ybAkCvKXFKYfYMttt4M2CP0i87ETU88sdCDwXW0eX85pVFc6huY44LJiuscHU8JYsUBzJZWcusTOpTHjEgzNwWrsGMfepeDu280aDPiZs/xI98F21MOMcyIwEkLje/FooxgHq9SBDCqiCjKZm8Ua17v5mS0kdDbVUZ7pVJssGOHA1SvMqGESoHwOKqACACcZSABy2SyMNFaBxbGvTUycPBqWF0mpKiwRAjDcpLe8CAhyD3yLULHUboV/721CwD4YykZK1tKu9+KjgrJblO91kgvWx68428fNjCnOqHa+pPUSCaHF8mEdeBrW76laqNv+dRT+TJyHRNjbdbNzlJAXeA3w+jWNfUHabIQ4yD/aNQKaGF/4eevU9FT1aeVzmp0SapeFv5meiH1VlFCpKoo8VqbLj3fNCaK+bH8LQfTO7/SOPOvYa31drpyT9Ws76G+Yj7oUBsCvBe8EjBaeI4blrhH0b/nqR1rvC5h9bxhpktM7WbbmaGEC+huC5CueDM8p6xKnI9enMprpepk5aGElafh4HzxORuzQszy2SLyR4FMpK+ljfvO+yVEVmoGlGAE=',
              'thinking': 'The user wants me to write and run a Python script '
                          "that prints 'hello world'. I can do this by:\n"
                          '1. Using the text_editor tool to create a Python '
                          'script file\n'
                      

### File Read Tools

In [5]:
import base64
from agent_base.core.types import DocumentContent

doc_text = "The speed of light is approximately 299,792,458 meters per second."
doc_b64 = base64.b64encode(doc_text.encode()).decode()

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type="text/plain", media_id="doc_001", source_type="base64", data=doc_b64),
        TextContent(text="What speed is mentioned in this document?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.to_dict())

2026-03-02T09:42:51.327696Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:42:51.328878Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=1 delay=1.00s func=_create
2026-03-02T09:42:52.676478Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:42:52.679006Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=2 delay=2.00s func=_create
2026-03-02T09:42:55.004618Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:42:55.005872Z [error    ] retries_exhausted              [agent_base.providers.anthropic.retry] func=_create max_retries=3


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "messages.0.content.0.document.source.base64.media_type: Input should be 'application/pdf'"}, 'request_id': 'req_011CYe33EdoLvqkHgv1f9jMr'}

### Container Uploads

In [6]:
from agent_base.core.types import AttachmentContent

file_content = b"print('Hello from uploaded file!')"
file_response = await client.beta.files.upload(file=("script.py", file_content, "text/x-python"))

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        AttachmentContent(
            filename="script.py", media_type="text/x-python",
            media_id=file_response.id, source_type="file_id", data=file_response.id,
        ),
        TextContent(text="Read the uploaded file and describe what it does."),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        beta_headers=["files-api-2025-04-14"],
    )
)
pprint(res.to_dict())

2026-03-02T09:43:00.847400Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/files?beta=true "HTTP/1.1 200 OK" [httpx]
2026-03-02T09:43:01.211943Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:01.213049Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=1 delay=1.00s func=_create
2026-03-02T09:43:02.538980Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:02.539680Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=2 delay=2.00s func=_create
2026-03-02T09:43:04.899097Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:04.900278Z [error    ] retries_exhausted              [agent_base.providers.anthropic.retry] func=_create max_retries=3


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages: container_upload blocks can only be used when a code execution tool is enabled'}, 'request_id': 'req_011CYe33xp52svrt4Rv1gj3y'}

### Web Search with citations

In [7]:
res = await anthropic_provider.generate(
    system_prompt="Be brief. Cite your sources.",
    messages=[Message.user("What is the population of Tokyo?")],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(
        thinking_tokens=10000,
        server_tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )
)
pprint(res.to_dict())

for block in res.content:
    if hasattr(block, 'raw') and block.raw:
        citations = getattr(block.raw, 'citations', None)
        if citations:
            pprint({"citations": [c.model_dump() for c in citations]})

2026-03-02T09:43:11.010267Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 200 OK" [httpx]


{'content': [{'content_block_type': 'thinking',
              'kwargs': {},
              'signature': 'Eq0ICkYICxgCKkDE2WzIhnDmXxC9UMyKtwzz2c5kpVxqlsNBmhvzNcgEIUGBhYTFBSONcomFVwISDrmbzpdkaKn23ijOW0QNRdXFEgz5ktpMj6EHWj5ah3gaDD01H3BbTG9LcxElUSIwg9OQIsP6O2LcyLc00YCHdZEjv6N4aLauWH2QVtNTDS3CMOfYSY+2B/MT5oTjC2w3KpQHqUjrb9xhL+zy+jNjyQjej/U8H8u9KtQqBr/vgjDDIKXbAzn/EyjkEZikyJZPHnM4a9XGftvFwM3vk+7Qb6VtTMBoplNvjy4Q85z5wBlGP/B2NivaXee1NXqzyvEdd14f6Udb+FuV0m95Sb7l1WxEBaRFOPreslTaGTdSWVKE72npM7+MzFyhWKgCRaf/4JWpSX5/VrD1vZhrxtuz+EKDaX2ZqL9sEAt9iH4kMEaualgdmiGoZ4ZV7wNiHXdNkZclILO822KL87jbF6PtlJbPcqEgleA3xC8XV35e7jVOpz++j/LLYyGKGp0L6mYcS8S1CgtsNDIuIFUUmdG+UmpFmkBfyEFuH1JcQgYfjg8eIm4MJDAytlbQDDGMBZFITeLYH44V9Lk8ouk0aGBkpcTFg8FVBBPpeMsOBKBKLf37r5vsGpdCLtyZxHoGm4KJAFWjdNaMVOMWBHByaJKZn3y8cgGmlQzPuwNNs+YNjY0HmpY1NxE4IGYbze0g0RaxeIoAP0v9g/IXjXEet6P6PFLPwb8oZoc56ni1Pabc8u00t6B52cEekLT07V2lGpMHexZZSb93Pa5Rug6PlfQkCHeEhpRq1mgWVp2fSkDjU+GI+hDe28XHpQch3wbKAo+oo6K32dm3rAqNs8Ouvmo9rwJ7NxG3QhddqnZsJTcbDqdj7w/dS6vK

### Document citations

In [8]:
doc_text = "Albert Einstein was born on March 14, 1879 in Ulm, Germany. He developed the theory of relativity."
doc_b64 = base64.b64encode(doc_text.encode()).decode()

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type="text/plain", media_id="doc_einstein", source_type="base64", data=doc_b64),
        TextContent(text="Where was Einstein born?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.to_dict())

2026-03-02T09:43:13.962090Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:13.962759Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=1 delay=1.00s func=_create
2026-03-02T09:43:15.378348Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:15.378988Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=2 delay=2.00s func=_create
2026-03-02T09:43:17.702735Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:17.703370Z [error    ] retries_exhausted              [agent_base.providers.anthropic.retry] func=_create max_retries=3


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "messages.0.content.0.document.source.base64.media_type: Input should be 'application/pdf'"}, 'request_id': 'req_011CYe34ug5x8coZw1ZY7GHH'}

### Content Block Citations

In [9]:
doc1_b64 = base64.b64encode(b"Section A: Python is a programming language created by Guido van Rossum.").decode()
doc2_b64 = base64.b64encode(b"Section B: JavaScript was created by Brendan Eich and runs in browsers.").decode()

res = await anthropic_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type="text/plain", media_id="doc_a", source_type="base64", data=doc1_b64),
        DocumentContent(media_type="text/plain", media_id="doc_b", source_type="base64", data=doc2_b64),
        TextContent(text="Who created each language mentioned in the documents?"),
    ])],
    tool_schemas=[],
    model="claude-haiku-4-5",
    llm_config=AnthropicLLMConfig(thinking_tokens=10000)
)
pprint(res.to_dict())

2026-03-02T09:43:21.300710Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:21.301422Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=1 delay=1.00s func=_create
2026-03-02T09:43:22.739555Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:22.740220Z [warning  ] retry                          [agent_base.providers.anthropic.retry] attempt=2 delay=2.00s func=_create
2026-03-02T09:43:25.061375Z [info     ] HTTP Request: POST https://api.anthropic.com/v1/messages?beta=true "HTTP/1.1 400 Bad Request" [httpx]
2026-03-02T09:43:25.062766Z [error    ] retries_exhausted              [agent_base.providers.anthropic.retry] func=_create max_retries=3


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': "messages.0.content.0.document.source.base64.media_type: Input should be 'application/pdf'"}, 'request_id': 'req_011CYe35T9vuCC7wVWyLBVx4'}